In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:
%%RecordEvent
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
import pandas as pd
# from tpch import load_lineitem, load_part, q14
DATA_ROOT = "/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}



In [4]:

### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)


Index(['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 'L_QUANTITY',
       'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG',
       'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 'L_RECEIPTDATE',
       'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT', 'L_DUMMY'],
      dtype='object')


In [5]:

### cell 1 ###

def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [6]:
# %%time
# ### cell 2 ###

# # Define promotion type filter
# p_type_like = "PROMO"

# # Build masks for ship date and promo parts
# ship_mask = lineitem.L_SHIPDATE.between("1994-03-01", "1994-04-01", inclusive="left")
# promo_keys = part.P_PARTKEY[part.P_TYPE.str.startswith(p_type_like)]

# # Compute line-level revenue once
# rev = lineitem.L_EXTENDEDPRICE * (1.0 - lineitem.L_DISCOUNT)

# # Total revenue for the period
# total_rev = rev[ship_mask].sum()

# # Promo revenue for the same period
# promo_rev = rev[ship_mask & lineitem.L_PARTKEY.isin(promo_keys)].sum()

# # Final metric
# total = promo_rev * 100 / total_rev

In [7]:
%%time
#original
startDate = pd.Timestamp("1994-03-01")
endDate = pd.Timestamp("1994-04-01")
p_type_like = "PROMO"
part_filtered = part.loc[:, ["P_PARTKEY", "P_TYPE"]]
lineitem_filtered = lineitem.loc[
    :, ["L_EXTENDEDPRICE", "L_DISCOUNT", "L_SHIPDATE", "L_PARTKEY"]
]
sel = (lineitem_filtered.L_SHIPDATE >= startDate) & (
    lineitem_filtered.L_SHIPDATE < endDate
)
flineitem = lineitem_filtered[sel]
jn = flineitem.merge(part_filtered, left_on="L_PARTKEY", right_on="P_PARTKEY")
jn["TMP"] = jn.L_EXTENDEDPRICE * (1.0 - jn.L_DISCOUNT)
total = jn[jn.P_TYPE.str.startswith(p_type_like)].TMP.sum() * 100 / jn.TMP.sum()
    

CPU times: user 990 ms, sys: 676 ms, total: 1.67 s
Wall time: 1.62 s


In [8]:
total

np.float64(16.604391231430338)

In [7]:
total

np.float64(16.604391231430338)

In [7]:
### cell 3 ###

total = total + 1